**Note: Try to avoid *GROUP BY* clause to solve the problems**

For the problems use the *Health Insurance Claim* dataset. You can get the details as well as the dataset from [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### **Problem 1:** What are the top 5 patients who claimed the highest insurance amounts?

### **Problem 2:** What is the average insurance claimed by patients based on the number of children they have?

### **Problem 3:** What is the highest and lowest claimed amount by patients in each region?

### **Problem 4:** What is the percentage of smokers in each age group?

### **Problem 5:** What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

### **Problem 6:** For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

### **Problem 7:** Show the patient with the highest BMI in each region and their respective rank.

### **Problem 8:** Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.

### **Problem 9:** For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

### **Problem 10:** For each patient, find the maximum BMI value among their next three records (ordered by age).

### **Problem 11:** For each patient, find the rolling average of the last 2 claims.

### **Problem 12:** Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

/* =========================================================
   WINDOW FUNCTION PRACTICE – INSURANCE DATA
   Database : MySQL
   Table    : homework.insurance_data
   ========================================================= */


/* =========================================================
   1️⃣ Problem 1
   Top 5 patients with highest total claimed amount
   ========================================================= */


/* ---------- Method 1 : Using GROUP BY ---------- */

SELECT
    PatientID,
    SUM(claim) AS claim_amt
FROM homework.insurance_data
GROUP BY PatientID
ORDER BY claim_amt DESC
LIMIT 5;



/* ---------- Method 2 : Using WINDOW FUNCTIONS ---------- */
/* NOTE:
   Your logic was correct in intent,
   but ranking must be done on aggregated claim per PatientID.
   Only syntax-level correction applied.
*/

SELECT *
FROM (
    SELECT
        PatientID,
        claim,
        DENSE_RANK() OVER (ORDER BY claim DESC) AS rnk
    FROM homework.insurance_data
) t
WHERE rnk <= 5;



/* =========================================================
   2️⃣ Problem 2
   Average insurance claimed based on number of children
   ========================================================= */


/* ---------- Method 1 : Using GROUP BY ---------- */

SELECT
    children,
    AVG(claim) AS avg_claim
FROM homework.insurance_data
GROUP BY children;



/* ---------- Method 2 : Using WINDOW FUNCTIONS ---------- */
/* Only formatting + syntax correction applied */

SELECT
    children,
    avg_claim
FROM (
    SELECT
        children,
        AVG(claim) OVER (PARTITION BY children) AS avg_claim,
        ROW_NUMBER() OVER (PARTITION BY children) AS row_num
    FROM homework.insurance_data
) t
WHERE row_num = 1;


/* =========================================================
   WINDOW FUNCTION PRACTICE – INSURANCE DATA
   Database : MySQL
   Table    : homework.insurance_data
   ========================================================= */


/* =========================================================
   3️⃣ Problem 3
   Highest and Lowest claimed amount in each region
   ========================================================= */

SELECT
    region,
    min_claim,
    max_claim
FROM (
    SELECT
        region,
        MIN(claim) OVER (PARTITION BY region) AS min_claim,
        MAX(claim) OVER (PARTITION BY region) AS max_claim,
        ROW_NUMBER() OVER (PARTITION BY region) AS row_num
    FROM homework.insurance_data
) t
WHERE t.row_num = 1;



/* =========================================================
   4️⃣ Problem 4
   (Incomplete – no query provided)
   ========================================================= */



/* =========================================================
   5️⃣ Problem 5
   Difference between claim amount and first claim
   ========================================================= */

SELECT
    *,
    claim - FIRST_VALUE(claim) OVER (
        ORDER BY PatientID
    ) AS difference
FROM homework.insurance_data;



/* =========================================================
   6️⃣ Problem 6
   Difference between claim and average claim
   based on number of children
   ========================================================= */

SELECT
    *,
    claim - AVG(claim) OVER (PARTITION BY children) AS diff
FROM homework.insurance_data;



/* =========================================================
   7️⃣ Problem 7
   Highest BMI in each region with ranking
   ========================================================= */


/* ---------- Method 1 : (Not Provided) ---------- */



/* ---------- Method 2 : Using Window Function ---------- */

SELECT *
FROM (
    SELECT
        *,
        RANK() OVER (PARTITION BY region ORDER BY bmi DESC) AS group_rank,
        RANK() OVER (ORDER BY bmi DESC) AS overall_rank
    FROM homework.insurance_data
) t
WHERE t.group_rank = 1;


/* =========================================================
   8️⃣ Problem 8
   Difference between claim amount and claim of patient
   with highest BMI in the same region
   ========================================================= */


/* ---------- Method 1 : Using FIRST_VALUE ---------- */

SELECT
    *,
    claim - FIRST_VALUE(claim) OVER (
        PARTITION BY region
        ORDER BY bmi DESC
    ) AS diff
FROM homework.insurance_data;



/* ---------- Method 2 : (Syntax was invalid, intent unclear)
   Kept commented to avoid breaking SQL execution
*/

/*
SELECT
    region,
    claim - (
        SELECT avg(claim),
               max(bmi) OVER (PARTITION BY region)
        FROM homework.insurance_data
    ) AS diff
FROM homework.insurance_data;
*/



/* =========================================================
   9️⃣ Problem 9
   Difference between claim and highest claim
   (same region + smoker)
   ========================================================= */

SELECT
    *,
    (MAX(claim) OVER (PARTITION BY region, smoker) - claim) AS claim_diff
FROM homework.insurance_data
ORDER BY claim_diff DESC;



/* =========================================================
   🔟 Problem 10
   Maximum BMI among next 3 records (ordered by age)
   ========================================================= */

SELECT
    *,
    MAX(bmi) OVER (
        ORDER BY age
        ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING
    ) AS bmi_diff
FROM homework.insurance_data;



/* =========================================================
   1️⃣1️⃣ Problem 11
   Rolling average of last 2 claims
   ========================================================= */

SELECT
    *,
    AVG(claim) OVER (
        ORDER BY PatientID
        ROWS BETWEEN 2 PRECEDING AND 1 PRECEDING
    ) AS rolling_avg
FROM homework.insurance_data;



/* =========================================================
   1️⃣2️⃣ Problem 12
   First claimed insurance value for male and female
   (Non-diabetic, BMI between 25 and 30)
   ========================================================= */

WITH filtered_data AS (
    SELECT *
    FROM homework.insurance_data
    WHERE diabetic = 'No'
      AND bmi BETWEEN 25 AND 30
)

SELECT
    region,
    gender,
    first_claim
FROM (
    SELECT
        *,
        FIRST_VALUE(claim) OVER (
            PARTITION BY region, gender
            ORDER BY age
        ) AS first_claim,
        ROW_NUMBER() OVER (
            PARTITION BY region, gender
            ORDER BY age
        ) AS row_num
    FROM filtered_data
) t
WHERE row_num = 1;